# Guía 3 - Arquitectura Big Data con Spark

In [2]:
#IMPORTACION DE LIBRERIAS

#Aqui se importan las librerias necesarias para manipular datos y las rutas

import pandas as pd
import sqlite3
import json
from pathlib import Path
import numpy as np
%pip install faker

print("Librerias importadas!!!!!!!!!!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 5.9 MB/s eta 0:00:00
Librerias importadas!!!!!!!!!!


In [33]:
# GUIA 3 - INTRODUCCION A PYSPARK

In [34]:
# INSTALACION DE PYSPARK

# Aqui se realiza la instalacion del paquete pyspark dentro del entorno
# de Google Colab para habilitar el uso del motor de procesamiento distribuido
%pip install pyspark

print("PySpark instalado :C")

PySpark instalado :C


In [35]:
# IMPORTACION DE LIBRERIAS

# Aqui se importan las librerias necesarias como SparkSession y functions de pyspark,
# ademas de pandas y pathlib para manejar las rutas y los cruces de informacion
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pathlib import Path
import pandas as pd

print("Librerias importadas ---------------")

Librerias importadas ---------------


In [36]:
# CREACION DE SESION DE SPARK

# Aqui se inicializa y configura la sesion local de Spark, estableciendo
# el nombre de la aplicacion y definiendo que se usaran todos los nucleos locales disponibles
spark = SparkSession.builder \
  .master("local[*]")\
  .appName("SmartCampusAI_Guia3")\
  .getOrCreate()

print(f"Spark Version: {spark.version}")
print(f"Paralelismo: {spark.sparkContext.defaultParallelism}")
print("Sesion de spark inicializada correctamente :C")

Spark Version: 4.0.4
Paralelismo: 2
Sesion de spark inicializada correctamente :C


In [37]:
# DEFINICION DE RUTAS DE TRABAJO

# Aqui se configuran las rutas de los directorios tanto para leer los datos procesados
# en la guia 2 como para guardar los resultados analiticos que genere Spark
BASE_DIR = Path("/content")
DATA_PROCESSED = BASE_DIR/"datos"/"procesados"
DATA_ANALYTICS = BASE_DIR/"datos"/"analiticos"

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
DATA_ANALYTICS.mkdir(parents=True, exist_ok=True)

print(f"Datos procesados: {DATA_PROCESSED}")
print(f"Datos analiticos: {DATA_ANALYTICS}")
print("Rutas definidas y carpetas creadas :C")

Datos procesados: /content/datos/procesados
Datos analiticos: /content/datos/analiticos
Rutas definidas y carpetas creadas :C


In [38]:
# CARGA DE DATASET LIMPIO

# Aqui se procede a leer el archivo csv limpio generado en la guia anterior y
# se carga en un dataframe de Spark infiriendo el esquema automaticamente
ruta_entrada = DATA_PROCESSED/"dataset_limpio_Smart_Campus.csv"

print(f"Leyendo: {ruta_entrada}")

df = spark.read\
  .option("header", True)\
  .option("inferSchema", True)\
  .csv(str(ruta_entrada))

print(f"Filas: {df.count()}")
print(f"Columnas: {len(df.columns)}")
print("Dataset cargado en Spark exitosamente :C")

Leyendo: /content/datos/procesados/dataset_limpio_Smart_Campus.csv
Filas: 10000
Columnas: 17
Dataset cargado en Spark exitosamente :C


In [39]:
# VERIFICACION DE ESQUEMA Y DATOS

# Aqui se imprime la estructura del dataframe inferida por Spark y
# se muestran las primeras filas para comprobar que los datos esten correctos
print("\nEsquema de datos: -------------:c")
df.printSchema()

print("\nPrimeras filas: ")
df.show(5, truncate= False)
print("Esquema y datos verificados :C")


Esquema de datos: -------------:c
root
 |-- id_registro: integer (nullable = true)
 |-- fecha: date (nullable = true)
 |-- id_recurso: integer (nullable = true)
 |-- hora_inicio: timestamp (nullable = true)
 |-- hora_fin: timestamp (nullable = true)
 |-- estudiantes_presentes: integer (nullable = true)
 |-- facultad: string (nullable = true)
 |-- horas_uso: double (nullable = true)
 |-- horas_estudiante_totales: double (nullable = true)
 |-- sede: string (nullable = true)
 |-- nombre_recurso: string (nullable = true)
 |-- edificio: string (nullable = true)
 |-- tipo_recurso: string (nullable = true)
 |-- capacidad_maxima: integer (nullable = true)
 |-- porcentaje_ocupacion: double (nullable = true)
 |-- estado: string (nullable = true)
 |-- prioridad: string (nullable = true)


Primeras filas: 
+-----------+----------+----------+-------------------+-------------------+---------------------+----------+---------+------------------------+-----------+------------------------------+-------

In [40]:
# INSPECCION DE PARTICIONES

# Aqui se verifica el numero de particiones iniciales en las que Spark
# ha dividido el dataframe al momento de realizar la lectura
print(f"\nParticiones iniciales: {df.rdd.getNumPartitions()}")
print("Inspeccion de particiones completada :C")


Particiones iniciales: 1
Inspeccion de particiones completada :C


In [41]:
# FILTRADO DE REGISTROS VALIDOS

# Aqui se filtran los registros del dataframe conservando unicamente
# aquellos donde los estudiantes presentes en el laboratorio son mayores a cero
df_validos = df.filter(F.col("estudiantes_presentes") > 0)

print(f"Registros despues de filtrar: {df_validos.count()}")
print(f"Registros eliminados: {df.count() - df_validos.count()}")
print("Filtrado de registros finalizado :C")

Registros despues de filtrar: 10000
Registros eliminados: 0
Filtrado de registros finalizado :C


In [42]:
# CREACION DE COLUMNA DERIVADA

# Aqui se crea una nueva columna llamada espacios_disponibles restando
# la cantidad de estudiantes presentes de la capacidad maxima del recurso
df_enriquecido = df_validos.withColumn(
    "espacios_disponibles",
    F.col("capacidad_maxima") - F.col("estudiantes_presentes")
)

print("Columna 'espacios_disponibles' creada: --------:c")
df_enriquecido.select("id_registro", "capacidad_maxima", "estudiantes_presentes", "espacios_disponibles").show(5)
print("Columna derivada calculada y agregada correctamente :C")

Columna 'espacios_disponibles' creada: --------:c
+-----------+----------------+---------------------+--------------------+
|id_registro|capacidad_maxima|estudiantes_presentes|espacios_disponibles|
+-----------+----------------+---------------------+--------------------+
|          1|              40|                   15|                  25|
|          2|              40|                   21|                  19|
|          3|              35|                   12|                  23|
|          4|              25|                   33|                  -8|
|          5|              30|                   32|                  -2|
+-----------+----------------+---------------------+--------------------+
only showing top 5 rows
Columna derivada calculada y agregada correctamente :C


In [43]:
# AGRUPACION POR FACULTAD

# Aqui se agrupan los datos por facultad para calcular el uso total de horas,
# la ocupacion promedio y el total de sesiones registradas utilizando funciones de Spark
resumen_facultad = df_enriquecido.groupBy("facultad").agg(
  F.sum("horas_estudiante_totales").alias("uso_total_horas"),
  F.avg("porcentaje_ocupacion").alias("ocupacion_promedio"),
  F.count("*").alias("sesiones_registradas")
).orderBy(F.desc("uso_total_horas"))

print("Uso por facultad: -----------------:c")
resumen_facultad.show(truncate=False)
print("Agrupacion por facultad completada :C")

Uso por facultad: -----------------:c
+----------+---------------+------------------+--------------------+
|facultad  |uso_total_horas|ocupacion_promedio|sesiones_registradas|
+----------+---------------+------------------+--------------------+
|Ingenieria|270121.0       |70.61157076205309 |5144                |
|Diseño    |150477.0       |70.61476486860312 |2892                |
|Ciencias  |102524.0       |70.49093177189411 |1964                |
+----------+---------------+------------------+--------------------+

Agrupacion por facultad completada :C


In [44]:
# AGRUPACION POR TIPO DE RECURSO

# Aqui se realiza una agregacion similar pero agrupando por tipo de recurso
# para obtener metricas de uso especificas de laboratorios, infraestructuras, etc.
resumen_recurso = df_enriquecido.groupBy("tipo_recurso").agg(
  F.sum("horas_estudiante_totales").alias("uso_total_horas"),
  F.count("*").alias("sesiones_registradas")
).orderBy(F.desc("uso_total_horas"))

print("Uso por tipo de recurso: -----------------:c")
resumen_recurso.show(truncate=False)
print("Agrupacion por recurso completada :C")

Uso por tipo de recurso: -----------------:c
+---------------+---------------+--------------------+
|tipo_recurso   |uso_total_horas|sesiones_registradas|
+---------------+---------------+--------------------+
|Laboratorio    |366302.0       |7000                |
|Infraestructura|79139.0        |1500                |
|Equipamiento   |77681.0        |1500                |
+---------------+---------------+--------------------+

Agrupacion por recurso completada :C


In [45]:
# CREACION DE DIMENSION Y CRUCE DE DATOS

# Aqui se define un nuevo dataframe categorico sobre la criticidad de los recursos
# y se une (join) con nuestro dataframe principal usando el tipo_recurso como llave
tipos_rec = [
    ("Laboratorio", "Fisico", "Alta"),
    ("Equipamiento", "Hardware", "Media"),
    ("Infraestructura", "Servidor", "Critica")
]

df_tipos = spark.createDataFrame(tipos_rec, ["tipo_recurso", "categoria_recurso", "criticidad"])

print("Tabla de dimension de recursos: -----------------:c")
df_tipos.show()

df_con_recurso = df_enriquecido.join(df_tipos, on="tipo_recurso", how="left")

print(f"Filas antes del join: {df_enriquecido.count()}")
print(f"Filas despues del join: {df_con_recurso.count()}")

print("Datos combinados con detalles del recurso: -----------------:c")
df_con_recurso.select("id_registro", "tipo_recurso", "categoria_recurso", "criticidad", "horas_estudiante_totales").show(5)
print("Cruce de datos con tabla de dimension completado :C")

Tabla de dimension de recursos: -----------------:c
+---------------+-----------------+----------+
|   tipo_recurso|categoria_recurso|criticidad|
+---------------+-----------------+----------+
|    Laboratorio|           Fisico|      Alta|
|   Equipamiento|         Hardware|     Media|
|Infraestructura|         Servidor|   Critica|
+---------------+-----------------+----------+

Filas antes del join: 10000
Filas despues del join: 10000
Datos combinados con detalles del recurso: -----------------:c
+-----------+---------------+-----------------+----------+------------------------+
|id_registro|   tipo_recurso|categoria_recurso|criticidad|horas_estudiante_totales|
+-----------+---------------+-----------------+----------+------------------------+
|          1|Infraestructura|         Servidor|   Critica|                    30.0|
|          2|Infraestructura|         Servidor|   Critica|                    63.0|
|          6|   Equipamiento|         Hardware|     Media|                   

In [46]:
# CREACION DE VISTA TEMPORAL

# Aqui se registra el dataframe procesado como una vista temporal en memoria
# para permitir realizar consultas directamente utilizando lenguaje SQL
df_con_recurso.createOrReplaceTempView("campus_analiticas")
print("Vista temporal 'campus_analiticas' creada exitosamente :C")

Vista temporal 'campus_analiticas' creada exitosamente :C


In [47]:
# EJECUCION DE CONSULTA SQL

# Aqui se ejecuta una consulta SQL nativa sobre la vista temporal para
# extraer un resumen analitico del uso agrupado por cada tipo de recurso
consulta = """
  SELECT
    tipo_recurso,
    COUNT(*) AS sesiones_registradas,
    ROUND(SUM(horas_estudiante_totales), 2) AS uso_total_horas,
    ROUND(AVG(porcentaje_ocupacion), 2) AS ocupacion_promedio
  FROM campus_analiticas
  GROUP BY tipo_recurso
  ORDER BY uso_total_horas DESC
"""

resultado_sql = spark.sql(consulta)
print("Consulta SQL - Resumen por recurso: -----------------:c")
resultado_sql.show(truncate=False)
print("Consulta SQL ejecutada correctamente :C")

Consulta SQL - Resumen por recurso: -----------------:c
+---------------+--------------------+---------------+------------------+
|tipo_recurso   |sesiones_registradas|uso_total_horas|ocupacion_promedio|
+---------------+--------------------+---------------+------------------+
|Laboratorio    |7000                |366302.0       |70.41             |
|Infraestructura|1500                |79139.0        |64.24             |
|Equipamiento   |1500                |77681.0        |77.78             |
+---------------+--------------------+---------------+------------------+

Consulta SQL ejecutada correctamente :C


In [48]:
# INSPECCION DEL PLAN DE EJECUCION

# Aqui se muestra el plan fisico y logico que Spark ha diseñado por debajo
# para resolver y optimizar la consulta SQL realizada previamente
print("\nPlan de ejecucion formateado: -----------:c")
resultado_sql.explain("formatted")
print("Plan de ejecucion desplegado con exito :C")


Plan de ejecucion formateado: -----------:c
== Physical Plan ==
AdaptiveSparkPlan (17)
+- Sort (16)
   +- Exchange (15)
      +- HashAggregate (14)
         +- HashAggregate (13)
            +- Project (12)
               +- SortMergeJoin LeftOuter (11)
                  :- Sort (5)
                  :  +- Exchange (4)
                  :     +- Project (3)
                  :        +- Filter (2)
                  :           +- Scan csv  (1)
                  +- Sort (10)
                     +- Exchange (9)
                        +- Project (8)
                           +- Filter (7)
                              +- Scan ExistingRDD (6)


(1) Scan csv 
Output [4]: [estudiantes_presentes#22, horas_estudiante_totales#25, tipo_recurso#29, porcentaje_ocupacion#31]
Batched: false
Location: InMemoryFileIndex [file:/content/datos/procesados/dataset_limpio_Smart_Campus.csv]
PushedFilters: [IsNotNull(estudiantes_presentes), GreaterThan(estudiantes_presentes,0)]
ReadSchema: struct<estudian

In [49]:
# REPARTICION DE DATOS

# Aqui se evaluan las particiones actuales del dataframe y se obliga a Spark
# a reparticionarlo en 4 bloques para optimizar el rendimiento de la escritura
print(f"Particiones actuales {df_con_recurso.rdd.getNumPartitions()}")

df_particionado = df_con_recurso.repartition(4)
print(f"Nuevas particiones: {df_particionado.rdd.getNumPartitions()}")
print("Reparticion de datos completada :C")

Particiones actuales 1
Nuevas particiones: 4
Reparticion de datos completada :C


In [50]:
# GUARDADO EN FORMATO PARQUET

# Aqui se guarda el dataframe procesado en formato parquet dentro de la carpeta
# analitica, el cual es un formato de almacenamiento altamente optimizado
ruta_parquet = DATA_ANALYTICS/"campus_analiticas_parquet"

df_particionado.write \
  .mode("overwrite") \
  .parquet(str(ruta_parquet))

print(f"Parquet guardado en: {ruta_parquet}")
print("Guardado en formato Parquet finalizado :C")

Parquet guardado en: /content/datos/analiticos/campus_analiticas_parquet
Guardado en formato Parquet finalizado :C


In [51]:
# LECTURA DE ARCHIVO PARQUET

# Aqui se lee nuevamente el archivo parquet recien creado desde el almacenamiento
# para verificar que la informacion se ha persistido correctamente
df_leido = spark.read.parquet(str(ruta_parquet))

print(f"Filas leidas desde Parquet: {df_leido.count()}")
print("Primeras filas: ------------:c")
df_leido.show(5, truncate=False)
print("Lectura de Parquet completada :C")

Filas leidas desde Parquet: 10000
Primeras filas: ------------:c
+------------+-----------+----------+----------+-------------------+-------------------+---------------------+----------+---------+------------------------+-----------+------------------------------+----------+----------------+--------------------+---------+---------+--------------------+-----------------+----------+
|tipo_recurso|id_registro|fecha     |id_recurso|hora_inicio        |hora_fin           |estudiantes_presentes|facultad  |horas_uso|horas_estudiante_totales|sede       |nombre_recurso                |edificio  |capacidad_maxima|porcentaje_ocupacion|estado   |prioridad|espacios_disponibles|categoria_recurso|criticidad|
+------------+-----------+----------+----------+-------------------+-------------------+---------------------+----------+---------+------------------------+-----------+------------------------------+----------+----------------+--------------------+---------+---------+--------------------+--------

In [52]:
# AGREGACION UTILIZANDO PANDAS

# Aqui se realiza la misma agrupacion inicial por facultad pero utilizando la
# libreria Pandas pura para posteriormente comparar los resultados con Spark
df_pd = pd.read_csv(DATA_PROCESSED/"dataset_limpio_Smart_Campus.csv")

pd_facultad = df_pd.groupby("facultad")["horas_estudiante_totales"].sum().reset_index()
pd_facultad.columns = ["facultad", "uso_total_horas"]

print("Pandas uso por facultad: -----------------:c")
print(pd_facultad)
print("Agregacion con Pandas ejecutada :C")

Pandas uso por facultad: -----------------:c
     facultad  uso_total_horas
0    Ciencias         102524.0
1      Diseño         150477.0
2  Ingenieria         270121.0
Agregacion con Pandas ejecutada :C


In [53]:
# CONVERSION DE RESULTADOS DE SPARK A PANDAS

# Aqui se toma el dataframe resumido de Spark creado anteriormente y se convierte
# explicitamente a un dataframe de Pandas para facilitar la integracion
sp_facultad = resumen_facultad.select("facultad", "uso_total_horas").orderBy("facultad").toPandas()
print("\nSpark - uso por facultad: ------------------:c")
print(sp_facultad)
print("Conversion de Spark a Pandas finalizada :C")


Spark - uso por facultad: ------------------:c
     facultad  uso_total_horas
0    Ciencias         102524.0
1      Diseño         150477.0
2  Ingenieria         270121.0
Conversion de Spark a Pandas finalizada :C


In [54]:
# COMPARACION DE RESULTADOS

# Aqui se cruzan los dataframes resultantes de Pandas y Spark haciendo un merge
# para validar que los calculos agrupados coinciden en ambas herramientas
print("\nComparacion Pandas vs Spark")

comparacion = pd.merge(pd_facultad, sp_facultad, on="facultad", suffixes=('_pandas', '_spark'))
print(comparacion)
print("Comparacion completada: Los resultados son consistentes entre pandas y spark :C")


Comparacion Pandas vs Spark
     facultad  uso_total_horas_pandas  uso_total_horas_spark
0    Ciencias                102524.0               102524.0
1      Diseño                150477.0               150477.0
2  Ingenieria                270121.0               270121.0
Comparacion completada: Los resultados son consistentes entre pandas y spark :C


In [55]:
# REPORTE FINAL DE ARQUITECTURA

# Aqui se imprime un reporte consolidado por consola que resume todas las metricas
# del procesamiento y arquitectura elaborada durante esta guia
print("\nREPORTE DE ARQUITECTURA - GUÍA 3")
print("=" * 50)
print(f"Versión de Spark: {spark.version}")
print(f"Dataset de entrada: dataset_limpio_Smart_Campus.csv")
print(f"Filas procesadas: {df_enriquecido.count()}")
print(f"Columnas originales: {len(df.columns)}")
print(f"Columnas finales: {len(df_particionado.columns)}")
print(f"Particiones iniciales: {df.rdd.getNumPartitions()}")
print(f"Particiones finales: {df_particionado.rdd.getNumPartitions()}")
print(f"Salida Parquet: {ruta_parquet}")
print(f"Formato: Parquet")
print("=" * 50)
print("Reporte de arquitectura generado :C")


REPORTE DE ARQUITECTURA - GUÍA 3
Versión de Spark: 4.0.4
Dataset de entrada: dataset_limpio_Smart_Campus.csv
Filas procesadas: 10000
Columnas originales: 17
Columnas finales: 20
Particiones iniciales: 1
Particiones finales: 4
Salida Parquet: /content/datos/analiticos/campus_analiticas_parquet
Formato: Parquet
Reporte de arquitectura generado :C


In [56]:
# CIERRE DE SESION SPARK

# Aqui se finaliza y cierra de forma segura la sesion de Spark para liberar
# el consumo de memoria y recursos del entorno de Google Colab
spark.stop()
print("Sesion spark finalizada: --------------:c")

Sesion spark finalizada: --------------:c


In [57]:
# DESCARGA DE ARCHIVOS PARQUET

# Aqui se comprimen los archivos analiticos generados en formato parquet dentro
# de un archivo zip de forma automatica para descargarlos a la maquina local
!zip -r /content/campus_analiticas_parquet.zip /content/datos/analiticos/campus_analiticas_parquet/

from google.colab import files
files.download("/content/campus_analiticas_parquet.zip")

print("Proceso de descarga de Parquet completado! :C")

  adding: content/datos/analiticos/campus_analiticas_parquet/ (stored 0%)
  adding: content/datos/analiticos/campus_analiticas_parquet/_SUCCESS (stored 0%)
  adding: content/datos/analiticos/campus_analiticas_parquet/._SUCCESS.crc (stored 0%)
  adding: content/datos/analiticos/campus_analiticas_parquet/.part-00000-c51a6ed4-e3b9-473a-8fd8-4e1b7d8715d4-c000.snappy.parquet.crc (stored 0%)
  adding: content/datos/analiticos/campus_analiticas_parquet/part-00002-c51a6ed4-e3b9-473a-8fd8-4e1b7d8715d4-c000.snappy.parquet (deflated 29%)
  adding: content/datos/analiticos/campus_analiticas_parquet/.part-00003-c51a6ed4-e3b9-473a-8fd8-4e1b7d8715d4-c000.snappy.parquet.crc (stored 0%)
  adding: content/datos/analiticos/campus_analiticas_parquet/part-00001-c51a6ed4-e3b9-473a-8fd8-4e1b7d8715d4-c000.snappy.parquet (deflated 29%)
  adding: content/datos/analiticos/campus_analiticas_parquet/part-00000-c51a6ed4-e3b9-473a-8fd8-4e1b7d8715d4-c000.snappy.parquet (deflated 29%)
  adding: content/datos/analitico

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Proceso de descarga de Parquet completado! :C


In [58]:
# DESCARGA DE TODOS LOS ARCHIVOS

# Aqui se genera un archivo zip consolidado final que contiene todos los recursos
# procesados, analiticos y crudos de esta guia para su almacenamiento
!zip -r /content/todos_archivos_guia3.zip /content/datos/analiticos/ /content/datos/procesados/dataset_limpio_Smart_Campus.csv

from google.colab import files
files.download("/content/todos_archivos_guia3.zip")

print("Todos los archivos consolidados listos para descargar! :C")

  adding: content/datos/analiticos/ (stored 0%)
  adding: content/datos/analiticos/campus_analiticas_parquet/ (stored 0%)
  adding: content/datos/analiticos/campus_analiticas_parquet/_SUCCESS (stored 0%)
  adding: content/datos/analiticos/campus_analiticas_parquet/._SUCCESS.crc (stored 0%)
  adding: content/datos/analiticos/campus_analiticas_parquet/.part-00000-c51a6ed4-e3b9-473a-8fd8-4e1b7d8715d4-c000.snappy.parquet.crc (stored 0%)
  adding: content/datos/analiticos/campus_analiticas_parquet/part-00002-c51a6ed4-e3b9-473a-8fd8-4e1b7d8715d4-c000.snappy.parquet (deflated 29%)
  adding: content/datos/analiticos/campus_analiticas_parquet/.part-00003-c51a6ed4-e3b9-473a-8fd8-4e1b7d8715d4-c000.snappy.parquet.crc (stored 0%)
  adding: content/datos/analiticos/campus_analiticas_parquet/part-00001-c51a6ed4-e3b9-473a-8fd8-4e1b7d8715d4-c000.snappy.parquet (deflated 29%)
  adding: content/datos/analiticos/campus_analiticas_parquet/part-00000-c51a6ed4-e3b9-473a-8fd8-4e1b7d8715d4-c000.snappy.parquet 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Todos los archivos consolidados listos para descargar! :C
